In [1]:
import gzip
import csv
import re
from src.files.files import get_files

snoRegions = {}

with get_files().get_renamed_files().get_annotations_file().open_or_recompute() as gzipFile:
    csvFile = csv.reader(gzipFile, delimiter="\t")

    for line in csvFile:
        if not re.search("snoRNA", line[8]):
            continue

        meta = line[8].split(";")

        transcript_id = None

        for metaEntry in meta:
            if not re.search("transcript_id", metaEntry):
                continue

            transcript_id = metaEntry.strip().split(" ")[1][1:-1]
            
            if transcript_id == "":
                continue

            snoRegions[transcript_id] = (transcript_id, line[3], line[4], line[6])

print(len(snoRegions))
        

44


In [2]:
import gzip
import csv
import re
from src.files.files import get_files

file_keys = ["exons", "introns", "5utr", "3utr", "cds"]
keys = ["exon", "intron", "5utr", "3utr", "cds"]

modified_sno_transcripts = {} 

for file_key, key in zip(file_keys, keys):
    print(key)

    with get_files().get_assembled_region_intersects_files().get_files_dict()[file_key].open_or_recompute() as file:
        csvFile = csv.reader(file, delimiter="\t")
    
        for line in csvFile:
            if not line[3][:-len("_" + str(key))] in snoRegions:
                continue

            modified_sno_transcripts[line[3]] = line
            print(line)

exon
['chr8', '33513542', '33513543', 'NR_003041.1_exon', '0', '+', '33513474', '33513578', '0', '1', '104,', '0,']
['chr8', '66922479', '66922480', 'NR_002598.1_exon', '0', '-', '66922473', '66922549', '0', '1', '76,', '0,']
intron
5utr
3utr
cds


In [3]:
from src.files.files import get_files
import csv

local_modified_sno_transcripts = {}

with get_files().get_assembled_region_local_intersects_assembled_files().get_exons_file().open_or_recompute() as file:
    csvFile = csv.reader(file, delimiter="\t")

    for line in csvFile:
        if not line[0] in modified_sno_transcripts:
            continue

        local_modified_sno_transcripts[line[0]] = line
        print(line)

['NR_003041.1_exon', '68']
['NR_002598.1_exon', '69']


In [4]:
from src.files.files import get_files
import csv
from fasta import FASTA

get_files().get_assembled_region_fasta_files().get_exons_file().open_or_recompute()

file = FASTA(get_files().get_assembled_region_fasta_files().get_exons_file().get_possibly_gzip_path())

for entry in file:
    entry_name = entry.name[:entry.name.find(":")]

    if not entry_name in local_modified_sno_transcripts:
        continue

    entry_modifications = list(map(int, local_modified_sno_transcripts[entry_name][1].split(",")))
        
    print(entry_name)
    print(entry_modifications)
    print()

    entry_rna_sequence = str(entry.seq).upper().replace("T", "U")

    print(entry_rna_sequence)
    print()

    modified_entry_rna_sequence = entry_rna_sequence

    for modification in entry_modifications:
        modified_entry_rna_sequence = modified_entry_rna_sequence[:modification] + "6" + modified_entry_rna_sequence[modification + 1:]

    print(modified_entry_rna_sequence)
    print()

NR_003041.1_exon
[68]

AUCCUUUUGUAGUUCAUGAGCGUGAUGAUUGGGUGUUCAUACGCUUGUGUGAGAUGUGCCACCCUUGAACCUUGUUACGACGUGGGCACAUUACCCGUCUGACC

AUCCUUUUGUAGUUCAUGAGCGUGAUGAUUGGGUGUUCAUACGCUUGUGUGAGAUGUGCCACCCUUGA6CCUUGUUACGACGUGGGCACAUUACCCGUCUGACC

NR_002598.1_exon
[69]

ACAAUGAUGACUUAAAUUACUUUUUGCCGUUUACCCAGCUGAGGUUGUCUUUGAAGAAAUAAUUUUAAGACUGAGA

ACAAUGAUGACUUAAAUUACUUUUUGCCGUUUACCCAGCUGAGGUUGUCUUUGAAGAAAUAAUUUUAAG6CUGAGA

